# 001 · Arithmetic and Assignment Operators

> **Syntax › Operators** · Python 3.14

Python's arithmetic operators look familiar, but several behave differently from C# in ways that cause real bugs: `/` always produces a float, `//` and `%` round towards negative infinity, and there is no `++`.

**In this notebook you will learn:**

- the arithmetic operators and how `/`, `//`, `%` and `**` differ from C#
- `divmod()` for quotient and remainder in one step
- augmented assignment (`+=` and friends) and why there is no `++` / `--`
- the walrus operator `:=` (assignment expressions)
- operator precedence, including a surprising case with `**`

## C# mental model

| Operation | C# | Python | Note |
|---|---|---|---|
| Add, subtract, multiply | `+ - *` | `+ - *` | same |
| Division | `7 / 2` → `3` (int) | `7 / 2` → `3.5` | Python `/` is **always** true division |
| Integer division | `7 / 2` (truncates) | `7 // 2` → `3` | floors, doesn't truncate |
| Remainder | `-7 % 2` → `-1` | `-7 % 2` → `1` | sign follows the **divisor** |
| Power | `Math.Pow(2, 10)` → `1024.0` | `2 ** 10` → `1024` | stays an `int` for int operands |
| Increment | `x++` / `++x` | `x += 1` | no `++` / `--` |
| Assign in expression | `(x = Read()) != null` | `(x := read()) is not None` | walrus `:=` |

Python `int`s also never overflow — they grow as large as memory allows *(see Data Types › 001 · Numbers)*.

## The basics: `+`, `-`, `*`

These behave as you'd expect. Mixing an `int` and a `float` produces a `float`, just like C#'s implicit widening.

In [ ]:
print(7 + 2, 7 - 2, 7 * 2)
print(7 + 2.0)          # int + float -> float: 9.0
print(-(-5), +5)        # unary minus and plus

Many operators also work on other types — for example `+` concatenates and `*` repeats strings and lists:

In [ ]:
print("ab" + "cd", "ab" * 3)
print([0] * 4)

## True division `/` — always a float

This is the single biggest difference. In C#, `/` between two integers is integer division. In Python 3, `/` is **always** true division and always returns a `float` — even when the result is a whole number.

```csharp
// C#
Console.WriteLine(7 / 2);     // 3
Console.WriteLine(7 / 2.0);   // 3.5
Console.WriteLine(10 / 2);    // 5 (int)
```

In [ ]:
print(7 / 2)       # 3.5  — not 3!
print(10 / 2)      # 5.0  — a float, even though it divides evenly
print(type(10 / 2).__name__)

⚠️ **Gotcha:** because `/` returns a float, you can't use its result directly as a list index or a `range()` bound:

In [ ]:
items = ["a", "b", "c", "d", "e"]

try:
    middle = items[len(items) / 2]
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

print(items[len(items) // 2])   # use floor division for indexes

Division by zero raises `ZeroDivisionError` for **both** ints and floats. (In C#, floating-point division by zero quietly gives `Infinity` or `NaN`.)

In [ ]:
for a, b in [(1, 0), (1.0, 0.0)]:
    try:
        print(a / b)
    except ZeroDivisionError as e:
        print(f"{type(e).__name__}: {e}")

## Floor division `//`

`//` divides and rounds **down towards negative infinity** (floor). For positive numbers this matches C#'s integer division, but for negative results it differs, because C# **truncates towards zero**.

```csharp
// C#
Console.WriteLine(7 / 2);     //  3
Console.WriteLine(-7 / 2);    // -3   (truncated towards zero)
```

In [ ]:
print(7 // 2)      #  3
print(-7 // 2)     # -4  (floored: -3.5 rounds DOWN to -4)
print(7.5 // 2)    #  3.0 — with a float operand the result is a float

If you need C#'s truncating behaviour, convert the true quotient with `int()`, which truncates towards zero (or use `math.trunc`):

In [ ]:
print(int(-7 / 2))   # -3, like C#

## Modulo `%`

Python's `%` is consistent with `//`: the result always has the **same sign as the divisor** (the right operand). C#'s `%` is a *remainder* whose sign follows the **dividend** (the left operand).

| Expression | C# | Python |
|---|---|---|
| `7 % 3` | `1` | `1` |
| `-7 % 3` | `-1` | `2` |
| `7 % -3` | `1` | `-2` |

In [ ]:
print(7 % 3)      #  1
print(-7 % 3)     #  2
print(7 % -3)     # -2

Python's invariant is `a == (a // b) * b + a % b`, and it holds for every sign combination:

In [ ]:
for a, b in [(7, 3), (-7, 3), (7, -3), (-7, -3)]:
    q, r = a // b, a % b
    print(f"{a:>3} // {b:>2} = {q:>3}   {a:>3} % {b:>2} = {r:>2}   check: {q * b + r == a}")

This makes `%` very convenient for wrap-around logic — a negative offset still lands inside the valid range, with no `+ n` fix-up that C# code usually needs:

In [ ]:
days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
today = 1                       # Tue
print(days[(today - 3) % 7])    # three days ago -> Sat (C# would give -2 here)

import math
print(math.fmod(-7, 3))         # -1.0 — C#-style remainder when you need it

## `divmod()` — quotient and remainder together

`divmod(a, b)` returns the tuple `(a // b, a % b)` in a single call (C#: `Math.DivRem`, which uses truncation semantics).

```csharp
// C#
var (q, r) = Math.DivRem(130, 60);   // (2, 10)
```

In [ ]:
minutes, seconds = divmod(130, 60)
print(f"{minutes} min {seconds} s")

print(divmod(-7, 2))    # (-4, 1) — floor semantics

## Exponentiation `**`

Python has a power **operator**. With integer operands and a non-negative exponent the result is an exact `int` (C#'s `Math.Pow` always returns a `double`). A negative exponent produces a float.

```csharp
// C#
double kb = Math.Pow(2, 10);                       // 1024.0
var big = BigInteger.Pow(2, 100);                  // needs System.Numerics
```

In [ ]:
print(2 ** 10)        # 1024 (int)
print(2 ** 100)       # exact — ints have arbitrary precision
print(2 ** -1)        # 0.5
print(9 ** 0.5)       # 3.0 — fractional exponent: square root
print(pow(2, 10, 1000))   # built-in pow() also supports modular exponentiation: 24

⚠️ **Gotcha:** `**` binds more tightly than unary minus, and it is **right-associative**:

In [ ]:
print(-2 ** 2)        # -4, parsed as -(2 ** 2)
print((-2) ** 2)      #  4
print(2 ** 3 ** 2)    # 512, parsed as 2 ** (3 ** 2), not (2 ** 3) ** 2 = 64

## Augmented assignment

Every binary operator has an augmented form: `+=`, `-=`, `*=`, `/=`, `//=`, `%=`, `**=`, plus the bitwise ones *(see Operators › 003 · Bitwise Operators)*.

```csharp
// C#
var total = 10;
total += 5;
total *= 2;
```

In [ ]:
total = 10
total += 5
total *= 2
total //= 3
total **= 2
total %= 7
print(total)

Note that `/=` turns an `int` variable into a `float`:

In [ ]:
count = 10
count /= 2
print(count, type(count).__name__)   # 5.0 float

⚠️ **Gotcha:** for mutable objects such as lists, `+=` modifies the object **in place** (like `AddRange`), while `x = x + y` creates a new object. With an alias, the difference is visible:

In [ ]:
a = [1, 2]
alias = a
a += [3]          # in-place: alias sees the change
print(alias)

b = [1, 2]
alias_b = b
b = b + [3]       # new list: alias_b still holds the old one
print(alias_b)

## No `++` or `--`

Python has no increment/decrement operators — write `x += 1`. Leaving them out is deliberate: `x += 1` is explicit, and there is no pre- vs post-increment subtlety to get wrong. Most C# uses of `i++` (loop counters) disappear anyway, because `for` loops iterate directly over items or a `range()` *(see Control Flow › 003 · For Loops and Range)*.

⚠️ **Gotcha:** `x++` is a syntax error, but `++x` is *valid* — it is just unary plus applied twice, and it does nothing:

In [ ]:
x = 5
++x               # +(+x) -> evaluates to 5, x is unchanged
--x               # -(-x) -> also 5
print(x)          # still 5

try:
    compile("x++", "<example>", "exec")
except SyntaxError as e:
    print(f"{type(e).__name__}: {e}")

## The walrus operator `:=`

In Python, `=` is a **statement**, not an expression — you can't write `if (x = f())`. Python 3.8 added the *assignment expression* `:=` (nicknamed the walrus operator) for when you need to assign and use a value in one expression.

```csharp
// C# — assignment is an expression
string? line;
while ((line = reader.ReadLine()) != null)
    Console.WriteLine(line);
```

In [ ]:
import io

reader = io.StringIO("first\nsecond\nthird\n")
while (line := reader.readline()):      # assign AND test
    print(line.strip())

In [ ]:
values = [3, 8, 1, 9, 4]

if (n := len(values)) > 3:
    print(f"list is long ({n} items)")

# reuse an expensive result inside a comprehension
results = [y for x in values if (y := x * x) > 10]
print(results)

`:=` can't be used as a plain statement (`x := 5` on its own line is a `SyntaxError`) — use `=` there. Parentheses are usually required around it. Use it sparingly: it is great in `while` loops and comprehensions, but plain assignments are clearer elsewhere.

## Operator precedence

Python's precedence is close to C#'s. From highest to lowest (for the operators covered so far):

| Precedence | Operators |
|---|---|
| highest | `**` |
| | unary `+x`, `-x`, `~x` |
| | `*`, `/`, `//`, `%` |
| | `+`, `-` |
| | `<<`, `>>`, then `&`, then `^`, then `\|` |
| | comparisons: `<`, `==`, `in`, `is`, … |
| | `not`, then `and`, then `or` |
| | conditional expression `a if c else b` |
| lowest | `:=` |

When in doubt, add parentheses — they cost nothing and help the reader.

In [ ]:
print(2 + 3 * 4)        # 14
print((2 + 3) * 4)      # 20
print(10 - 4 - 3)       # 3  — left-associative: (10 - 4) - 3
print(-3 ** 2)          # -9 — ** before unary minus
print(17 // 5 * 5 + 17 % 5)   # 17 — // * % share a level, left to right

## Summary

| Task | C# | Python |
|---|---|---|
| True division | `7 / 2.0` | `7 / 2` → `3.5` |
| Integer division | `7 / 2` (truncates) | `7 // 2` (floors) |
| Remainder | `a % b` (sign of `a`) | `a % b` (sign of `b`); `math.fmod` for C# style |
| Quotient + remainder | `Math.DivRem(a, b)` | `divmod(a, b)` |
| Power | `Math.Pow(a, b)` | `a ** b` / `pow(a, b)` |
| Increment | `x++` | `x += 1` |
| Assign in expression | `(x = f()) != null` | `(x := f()) is not None` |
| Divide by zero (float) | `Infinity` / `NaN` | `ZeroDivisionError` |

## Exercises

1. Convert `7384` seconds into hours, minutes and seconds using `divmod()` and print `2h 3m 4s`.
2. Write `c_sharp_div(a, b)` and `c_sharp_mod(a, b)` that reproduce C#'s integer `/` and `%` for ints of any sign. Check them with `(-7, 2)` and `(7, -2)`.
3. Predict, then check: `-2 ** 2`, `2 ** -2`, `-7 // 2`, `-7 % 2`, `10 / 5`.
4. Use the walrus operator to keep reading chunks of 4 characters from `io.StringIO("abcdefghij")` with `.read(4)` until it returns an empty string, printing each chunk.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
# 1
minutes, seconds = divmod(7384, 60)
hours, minutes = divmod(minutes, 60)
print(f"{hours}h {minutes}m {seconds}s")      # 2h 3m 4s

# 2 — truncate the true quotient towards zero
def c_sharp_div(a, b):
    return int(a / b)          # fine for moderate ints; math.trunc for clarity

def c_sharp_mod(a, b):
    return a - b * c_sharp_div(a, b)

print(c_sharp_div(-7, 2), c_sharp_mod(-7, 2))   # -3 -1
print(c_sharp_div(7, -2), c_sharp_mod(7, -2))   # -3 1

# 3
print(-2 ** 2)   # -4
print(2 ** -2)   # 0.25
print(-7 // 2)   # -4
print(-7 % 2)    # 1
print(10 / 5)    # 2.0

# 4
import io
stream = io.StringIO("abcdefghij")
while chunk := stream.read(4):
    print(chunk)      # abcd, efgh, ij
```

</details>